In [1]:
!unzip -q fuzzer_colab.zip -d project_folder



In [2]:
!apt-get update
!apt-get install -y verilator


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InReleas

In [3]:
%%bash
cd project_folder/smaller_implementation/verilator_env
bash build_buggy.sh


=== Building Buggy ALU for Differential Testing ===
Build dir: /tmp/verilator_buggy_Tfo5dM


%Error: alu_buggy.v:15:5: Unknown verilator lint message code: 'UNUSEDSIGNAL', in '/*verilator lint_off UNUSEDSIGNAL*/'
   15 |     /*verilator lint_off UNUSEDSIGNAL*/ 
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
%Error: alu_buggy.v:17:5: Unknown verilator lint message code: 'UNUSEDSIGNAL', in '/*verilator lint_on UNUSEDSIGNAL*/'
   17 |     /*verilator lint_on UNUSEDSIGNAL*/ 
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
%Error: Exiting due to 2 error(s)


CalledProcessError: Command 'b'cd project_folder/smaller_implementation/verilator_env\nbash build_buggy.sh\n'' returned non-zero exit status 1.

In [6]:
import os

file_path = 'project_folder/smaller_implementation/verilator_env/alu_buggy.v'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    with open(file_path, 'w') as f:
        # Insert a generic lint off for UNUSED at the top of the module
        f.write('/* verilator lint_off UNUSED */\n')
        for line in lines:
            # Still remove the problematic specific code that caused the first error
            if 'UNUSEDSIGNAL' not in line:
                f.write(line)
    print(f'Applied broad UNUSED suppression to {file_path}')
else:
    print(f'File {file_path} not found.')

Applied broad UNUSED suppression to project_folder/smaller_implementation/verilator_env/alu_buggy.v


In [19]:
import os

verilog_file = 'project_folder/smaller_implementation/verilator_env/alu_buggy.v'
build_script = 'project_folder/smaller_implementation/verilator_env/build_buggy.sh'

# 1. Fix the Verilog file (remove the unsupported lint pragmas)
with open(verilog_file, 'r') as f:
    v_code = f.read()

v_code = v_code.replace('/* verilator lint_off UNUSEDSIGNAL */', '')
v_code = v_code.replace('/* verilator lint_on UNUSEDSIGNAL */', '')

with open(verilog_file, 'w') as f:
    f.write(v_code)

# 2. Restore the correct build script (with -Wno-lint added to ignore the unused signal warning)
build_sh_content = """#!/bin/bash
set -e

SCRIPT_DIR="$(cd "$(dirname "$0")" && pwd)"
BUILD_DIR=$(mktemp -d /tmp/verilator_buggy_XXXXXX)

echo "=== Building Buggy ALU for Differential Testing ==="
cp "$SCRIPT_DIR/alu_buggy.v" "$BUILD_DIR/"
cp "$SCRIPT_DIR/alu_wrapper.cpp" "$BUILD_DIR/"
cd "$BUILD_DIR"

# Build with -Wno-lint to ignore any unused signal warnings on Colab's Verilator
verilator -Wno-lint -Wno-DECLFILENAME --prefix Valu --cc alu_buggy.v

cd obj_dir
make -f Valu.mk CXXFLAGS="-fPIC" Valu__ALL.a verilated.o verilated_threads.o || make -f Valu.mk CXXFLAGS="-fPIC" Valu__ALL.a verilated.o
cd ..

g++ -fPIC -shared \\
    -I./obj_dir \\
    -I/usr/share/verilator/include \\
    -I/usr/share/verilator/include/vltstd \\
    alu_wrapper.cpp \\
    obj_dir/Valu__ALL.a \\
    obj_dir/verilated.o \\
    $(ls obj_dir/verilated_threads.o 2>/dev/null || echo "") \\
    -pthread -lpthread -latomic \\
    -o libalu_buggy.so

mkdir -p "$SCRIPT_DIR/obj_dir_buggy"
cp libalu_buggy.so "$SCRIPT_DIR/obj_dir_buggy/"
rm -rf "$BUILD_DIR"

echo "Buggy shared library built: obj_dir_buggy/libalu_buggy.so"
echo "=== Done ==="
"""

with open(build_script, 'w') as f:
    f.write(build_sh_content)

print("Files patched successfully! Now running the build script...")


Files patched successfully! Now running the build script...


In [20]:
%%bash
cd project_folder/smaller_implementation/verilator_env
bash build_buggy.sh


=== Building Buggy ALU for Differential Testing ===
/usr/bin/perl /usr/share/verilator/bin/verilator_includer -DVL_INCLUDE_OPT=include Valu.cpp Valu__Slow.cpp Valu__Syms.cpp > Valu__ALL.cpp
g++ -fPIC -I.  -MMD -I/usr/share/verilator/include -I/usr/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow      -Os -c -o Valu__ALL.o Valu__ALL.cpp
ar -cr Valu__ALL.a Valu__ALL.o
ranlib Valu__ALL.a
g++ -fPIC -I.  -MMD -I/usr/share/verilator/include -I/usr/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-sign-compare -Wno-uninitialized -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable -Wno-shadow      -Os -c -o verilated.o /usr/share/verilator/include/verilated.cpp
g++ -fPIC -I.  -MMD -I/usr/share/verilato

In file included from /usr/share/verilator/include/verilated_threads.cpp:18:
/usr/share/verilator/include/verilated_threads.h:48:17: error: ‘atomic’ in namespace ‘std’ does not name a template type
   48 |     static std::atomic<vluint64_t> s_yields;  // Statistics
      |                 ^~~~~~
/usr/share/verilator/include/verilated_threads.h:28:1: note: ‘std::atomic’ is defined in header ‘<atomic>’; did you forget to ‘#include <atomic>’?
   27 | #include <vector>
  +++ |+#include <atomic>
   28 | 
/usr/share/verilator/include/verilated_threads.h:66:10: error: ‘atomic’ in namespace ‘std’ does not name a template type
   66 |     std::atomic<vluint32_t> m_upstreamDepsDone;
      |          ^~~~~~
/usr/share/verilator/include/verilated_threads.h:66:5: note: ‘std::atomic’ is defined in header ‘<atomic>’; did you forget to ‘#include <atomic>’?
   66 |     std::atomic<vluint32_t> m_upstreamDepsDone;
      |     ^~~
/usr/share/verilator/include/verilated_threads.h: In static member function

In [21]:
!python project_folder/smaller_implementation/MILESTONE_6/parallel_fuzzer.py


  MILESTONE 6: Parallel Fuzzer (Multi-Worker)
  Spawning 4 workers...
[Worker 3] Starting...
[Worker 0] Starting...
[Worker 1] Starting...
[Worker 2] Starting...
[Worker 2] Ep 9: Found 2 new bugs! Total worker bugs: 2 | Unique classes: 1
[Worker 0] Ep 27: Found 1 new bugs! Total worker bugs: 1 | Unique classes: 1
[Worker 1] Ep 39: Found 10 new bugs! Total worker bugs: 10 | Unique classes: 1
[Worker 2] Ep 33: Found 2 new bugs! Total worker bugs: 4 | Unique classes: 1
[Worker 1] Ep 54: Found 2 new bugs! Total worker bugs: 12 | Unique classes: 1
[Main] Processed 1000 transitions. Queue size: 20000. Time: 14.5s
[Main] Processed 2000 transitions. Queue size: 20000. Time: 16.5s
[Main] Processed 3000 transitions. Queue size: 20000. Time: 18.5s
[Main] Processed 4000 transitions. Queue size: 20000. Time: 20.5s
[Main] Processed 5000 transitions. Queue size: 20000. Time: 22.4s
[Main] Processed 6000 transitions. Queue size: 20000. Time: 25.2s
[Main] Processed 7000 transitions. Queue size: 20000. T